In [5]:
import os
for dirname, _, filenames in os.walk('./SST2Data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/datasetSentences.txt
./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/datasetSplit.txt
./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/dictionary.txt
./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/original_rt_snippets.txt
./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/README.txt
./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/sentiment_labels.txt
./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/SOStr.txt
./SST2Data/stanfordSentimentTreebank/stanfordSentimentTreebank/STree.txt
./SST2Data/stanfordSentimentTreebankRaw/stanfordSentimentTreebankRaw/rawscores_exp12.txt
./SST2Data/stanfordSentimentTreebankRaw/stanfordSentimentTreebankRaw/README.txt
./SST2Data/stanfordSentimentTreebankRaw/stanfordSentimentTreebankRaw/sentlex_exp12.txt
./SST2Data/trainDevTestTrees_PTB/trees/dev.txt
./SST2Data/trainDevTestTrees_PTB/trees/test.txt
./SST2Data/trai

In [6]:
!python -m pip install pytreebank 
import pytreebank
import pandas as pd
import os

In [7]:
import pytreebank
import pandas as pd
import os

# Define the path where your .txt tree files are located
tree_path = './SST2Data/trainDevTestTrees_PTB/trees/'

def load_trees_to_df(path):
    data = []
    for split in ['train', 'dev', 'test']:
        full_path = os.path.join(path, f"{split}.txt")
        
        if not os.path.exists(full_path):
            print(f"Warning: {full_path} not found.")
            continue
            
        with open(full_path, 'r') as f:
            for line in f:
                # Create the tree object
                tree = pytreebank.create_tree_from_string(line)
                
                # FIX: Use .label instead of .root_value
                data.append({
                    "text": tree.to_lines()[0], # Get the full sentence
                    "label": tree.label,        # Get the sentiment (0-4)
                    "split": split
                })
    return pd.DataFrame(data)

# Run the loader
df = load_trees_to_df(tree_path)

# Split the data as described in the paper
train_df = df[df['split'] == 'train'] # ~8,544 examples
dev_df = df[df['split'] == 'dev']     # ~1,101 examples[cite: 1]
test_df = df[df['split'] == 'test']   # ~2,210 examples[cite: 1]

print(f"Success! Training rows: {len(train_df)}")
print(train_df) # Look at the first 5 rows to confirm

Success! Training rows: 8544
                                                   text  label  split
0     The Rock is destined to be the 21st Century 's...      3  train
1     The gorgeously elaborate continuation of `` Th...      4  train
2     Singer/composer Bryan Adams contributes a slew...      3  train
3     You 'd think by now America would have had eno...      2  train
4                  Yet the act is still charming here .      3  train
...                                                 ...    ...    ...
8539                                    A real snooze .      0  train
8540                                     No surprises .      1  train
8541  We 've seen the hippie-turned-yuppie plot befo...      3  train
8542  Her fans walked out muttering words like `` ho...      0  train
8543                                In this case zero .      1  train

[8544 rows x 3 columns]


In [8]:
class SentimentNode:
    def __init__(self, label, text=None, left=None, right=None):
        self.label = label      # Sentiment score (0-4)
        self.text = text        # Only 'leaf' nodes have text
        self.left = left        # Left child (SentimentNode)
        self.right = right      # Right child (SentimentNode)

    def is_leaf(self):
        return self.left is None and self.right is None

def find_balanced_split(s):
    depth = 0
    for i, ch in enumerate(s):
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
        if depth == 0:
            return i + 1
    return len(s)

def parse_sst_string(tokens):
    """
    Very basic recursive parser for (Label Text) or (Label Left Right)
    """
    # Remove the outer parentheses
    tokens = tokens.strip()[1:-1]
    
    # The first character is always the label
    label = int(tokens[0])
    content = tokens[2:].strip()
    
    # If there are no more parentheses, it's a leaf (a word)
    if '(' not in content:
        return SentimentNode(label=label, text=content)
    
    # Otherwise, find the split point between Left and Right children
    # This involves counting parentheses to find the balanced middle
    # (Simplified logic here for clarity)
    split_idx = find_balanced_split(content) 
    left_str = content[:split_idx]
    right_str = content[split_idx:]
    
    return SentimentNode(
        label=label, 
        left=parse_sst_string(left_str), 
        right=parse_sst_string(right_str)
    )

parse_sst_string("(3 (2 The) (4 movie))")